# Advanced Problems: Sentinel Values for Parameter Defaults

These problems focus on robust use of sentinel objects in Python function signatures.

## Learning Goals

By the end, you should be able to:

- distinguish “argument omitted” from “argument explicitly passed as `None`”
- design private sentinel objects safely
- avoid equality-based sentinel checks
- work with positional, keyword-only, and variadic parameters
- understand trade-offs between module-level sentinels, inline `object()` defaults, and custom sentinel classes
- preserve sentinel behavior across decorators, pickling, and APIs

## Setup

Run this cell before attempting the problems.

In [1]:
from __future__ import annotations

import functools
import inspect
import pickle
from dataclasses import dataclass, field

## Problem 1 — Tri-state argument detection

Write a function `classify_argument(value=...)` that returns:

- `"omitted"` if no argument was supplied
- `"explicit None"` if the caller supplied `None`
- `"value: <repr>"` for any other value

Requirements:

1. Do not use `None` as the default.
2. Use identity comparison, not equality comparison.
3. The following calls must work:

```python
classify_argument()
classify_argument(None)
classify_argument(0)
classify_argument("")
classify_argument(object())
```

In [2]:
# Your solution here

### Solution 1

A unique sentinel object lets us reserve `None` as a meaningful user value.

In [3]:
_MISSING = object()

def classify_argument(value=_MISSING):
    if value is _MISSING:
        return "omitted"
    if value is None:
        return "explicit None"
    return f"value: {value!r}"


assert classify_argument() == "omitted"
assert classify_argument(None) == "explicit None"
assert classify_argument(0) == "value: 0"
assert classify_argument("") == "value: ''"
assert classify_argument(object()).startswith("value: <object object at")

[classify_argument(), classify_argument(None), classify_argument(0), classify_argument("")]

['omitted', 'explicit None', 'value: 0', "value: ''"]

## Problem 2 — Why equality is dangerous

Create a class `AlwaysEqual` whose instances compare equal to every object.

Then show why this implementation is unsafe:

```python
BAD_SENTINEL = object()

def bad(value=BAD_SENTINEL):
    if value == BAD_SENTINEL:
        return "omitted"
    return "provided"
```

Finally, write a corrected version called `good`.

In [4]:
# Your solution here

### Solution 2

Sentinel checks should use `is`, not `==`. Equality can be user-controlled through `__eq__`.

In [5]:
class AlwaysEqual:
    def __eq__(self, other):
        return True

BAD_SENTINEL = object()

def bad(value=BAD_SENTINEL):
    if value == BAD_SENTINEL:
        return "omitted"
    return "provided"

GOOD_SENTINEL = object()

def good(value=GOOD_SENTINEL):
    if value is GOOD_SENTINEL:
        return "omitted"
    return "provided"


tricky = AlwaysEqual()

assert bad(tricky) == "omitted"       # incorrect
assert good(tricky) == "provided"     # correct
assert good() == "omitted"

bad(tricky), good(tricky), good()

('omitted', 'provided', 'omitted')

## Problem 3 — Multiple sentinel defaults without global names

Write a function:

```python
def audit(a=object(), b=object(), *, strict=object()):
    ...
```

It should return a dictionary indicating whether each argument was provided:

```python
{
    "a": True or False,
    "b": True or False,
    "strict": True or False
}
```

Requirements:

1. Do not bind the sentinels to separate global variables.
2. Retrieve the default objects from the function object itself.
3. Support positional and keyword-only defaults.

In [6]:
# Your solution here

### Solution 3

Positional defaults live in `function.__defaults__`; keyword-only defaults live in `function.__kwdefaults__`.

In [7]:
def audit(a=object(), b=object(), *, strict=object()):
    default_a, default_b = audit.__defaults__
    default_strict = audit.__kwdefaults__["strict"]

    return {
        "a": a is not default_a,
        "b": b is not default_b,
        "strict": strict is not default_strict,
    }


assert audit() == {"a": False, "b": False, "strict": False}
assert audit(10) == {"a": True, "b": False, "strict": False}
assert audit(b=None) == {"a": False, "b": True, "strict": False}
assert audit(strict=False) == {"a": False, "b": False, "strict": True}
assert audit(1, 2, strict=None) == {"a": True, "b": True, "strict": True}

audit(), audit(1), audit(b=None), audit(strict=False), audit(1, 2, strict=None)

({'a': False, 'b': False, 'strict': False},
 {'a': True, 'b': False, 'strict': False},
 {'a': False, 'b': True, 'strict': False},
 {'a': False, 'b': False, 'strict': True},
 {'a': True, 'b': True, 'strict': True})

## Problem 4 — Decorator pitfall with inline sentinels

The function below uses inline sentinels and introspects its own defaults:

```python
def parse(value=object()):
    default_value = parse.__defaults__[0]
    return "omitted" if value is default_value else "provided"
```

Now suppose it is decorated:

```python
def logger(fn):
    def wrapper(*args, **kwargs):
        print(f"calling {fn.__name__}")
        return fn(*args, **kwargs)
    return wrapper

parse = logger(parse)
```

Explain why the original introspection pattern can break or become fragile with decorators.

Then write a robust version using a module-level sentinel and `functools.wraps`.

In [8]:
# Your solution here

### Solution 4

The fragile part is that the function body refers to the global name `parse`. After decoration, that name may point to the wrapper, not the original function object. The wrapper may not have the same `__defaults__`, so introspection through the function name can fail.

A named sentinel avoids that dependency.

In [9]:
def logger(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        print(f"calling {fn.__name__}")
        return fn(*args, **kwargs)
    return wrapper


PARSE_MISSING = object()

@logger
def parse(value=PARSE_MISSING):
    return "omitted" if value is PARSE_MISSING else "provided"


assert parse() == "omitted"
assert parse(None) == "provided"
assert parse(0) == "provided"

parse(), parse(None)

calling parse
calling parse
calling parse
calling parse
calling parse


('omitted', 'provided')

## Problem 5 — Design a better repr for a sentinel

Using plain `object()` gives an unhelpful representation such as:

```python
<object object at 0x...>
```

Create a private sentinel called `_MISSING` whose `repr()` is exactly:

```python
<MISSING>
```

Requirements:

1. The sentinel should still be unique.
2. The sentinel should be checked with `is`.
3. Use it in a function `normalize(value=_MISSING)`:
   - omitted -> return `"using default"`
   - explicit `None` -> return `"clearing value"`
   - anything else -> return `"setting to <repr>"`

In [10]:
# Your solution here

### Solution 5

A tiny sentinel class improves debugging, documentation, and function signatures.

In [11]:
class _MissingType:
    def __repr__(self):
        return "<MISSING>"

_MISSING = _MissingType()

def normalize(value=_MISSING):
    if value is _MISSING:
        return "using default"
    if value is None:
        return "clearing value"
    return f"setting to {value!r}"


assert repr(_MISSING) == "<MISSING>"
assert normalize() == "using default"
assert normalize(None) == "clearing value"
assert normalize("abc") == "setting to 'abc'"

inspect.signature(normalize), normalize(), normalize(None), normalize("abc")

(<Signature (value=<MISSING>)>,
 'using default',
 'clearing value',
 "setting to 'abc'")

## Problem 6 — Sentinel values in dataclasses

You are building a configuration object where `timeout` has three meanings:

- omitted: inherit timeout from parent config
- `None`: disable timeout
- number: use that timeout

Write a dataclass `RequestConfig` with a `timeout` field that can distinguish all three cases.

Requirements:

1. Do not use `None` as the field default.
2. Provide a method `effective_timeout(parent_timeout)`:
   - omitted -> return `parent_timeout`
   - `None` -> return `None`
   - numeric value -> return that value
3. Add assertions for all three cases.

In [12]:
# Your solution here

### Solution 6

The same sentinel pattern works in dataclass fields. Use `default=...`, not `default_factory`, because we want one stable sentinel identity.

In [13]:
class _UnsetType:
    def __repr__(self):
        return "<UNSET>"

UNSET = _UnsetType()

@dataclass
class RequestConfig:
    timeout: object = field(default=UNSET)

    def effective_timeout(self, parent_timeout):
        if self.timeout is UNSET:
            return parent_timeout
        return self.timeout


assert RequestConfig().effective_timeout(30) == 30
assert RequestConfig(timeout=None).effective_timeout(30) is None
assert RequestConfig(timeout=5).effective_timeout(30) == 5

RequestConfig(), RequestConfig(timeout=None), RequestConfig(timeout=5)

(RequestConfig(timeout=<UNSET>),
 RequestConfig(timeout=None),
 RequestConfig(timeout=5))

## Problem 7 — API patch semantics

Implement `patch_user(current, *, name=..., email=..., active=...)`.

The function receives a dictionary such as:

```python
{"name": "Ada", "email": "ada@example.com", "active": True}
```

Each keyword-only parameter should behave as follows:

- omitted: leave that field unchanged
- `None`: explicitly set that field to `None`
- any other value: set the field to that value

Requirements:

1. Do not mutate the original dictionary.
2. Use a sentinel for omitted parameters.
3. All patch parameters must be keyword-only.
4. Unknown fields are not part of this problem; only patch the three specified fields.

In [14]:
# Your solution here

### Solution 7

This is a common production use case: patch/update APIs need to distinguish “not included” from “included with null.”

In [15]:
PATCH_MISSING = object()

def patch_user(current, *, name=PATCH_MISSING, email=PATCH_MISSING, active=PATCH_MISSING):
    updated = current.copy()

    if name is not PATCH_MISSING:
        updated["name"] = name

    if email is not PATCH_MISSING:
        updated["email"] = email

    if active is not PATCH_MISSING:
        updated["active"] = active

    return updated


user = {"name": "Ada", "email": "ada@example.com", "active": True}

assert patch_user(user) == user
assert patch_user(user) is not user
assert patch_user(user, email=None)["email"] is None
assert patch_user(user, active=False)["active"] is False
assert patch_user(user, name="Grace", email="grace@example.com") == {
    "name": "Grace",
    "email": "grace@example.com",
    "active": True,
}

user, patch_user(user, email=None), patch_user(user, active=False)

({'name': 'Ada', 'email': 'ada@example.com', 'active': True},
 {'name': 'Ada', 'email': None, 'active': True},
 {'name': 'Ada', 'email': 'ada@example.com', 'active': False})

## Problem 8 — Sentinel and `**kwargs`

Write a function `extract_option(kwargs, key, default=...)`.

It should:

- remove `key` from the dictionary if present
- return the removed value
- if `key` is absent and `default` was provided, return `default`
- if `key` is absent and `default` was omitted, raise `KeyError(key)`

Requirements:

1. Mutate `kwargs` by removing the key if present.
2. Allow `default=None` to be a real default.
3. Do not use `dict.pop(key, default)` directly with `None` as the sentinel.

In [16]:
# Your solution here

### Solution 8

This mirrors the design of APIs where “no default was provided” is different from “the default is `None`.”

In [17]:
NO_DEFAULT = object()

def extract_option(kwargs, key, default=NO_DEFAULT):
    if key in kwargs:
        return kwargs.pop(key)

    if default is not NO_DEFAULT:
        return default

    raise KeyError(key)


options = {"limit": 10, "verbose": False}

assert extract_option(options, "limit") == 10
assert "limit" not in options
assert extract_option(options, "missing", default=None) is None

try:
    extract_option(options, "also_missing")
except KeyError as ex:
    assert ex.args == ("also_missing",)
else:
    raise AssertionError("Expected KeyError")

options

{'verbose': False}

## Problem 9 — Pickle-safe sentinel singleton

Plain `object()` sentinels are excellent for in-process checks, but custom APIs sometimes need a sentinel that preserves identity after pickling and unpickling.

Create a singleton sentinel named `MISSING` such that:

```python
pickle.loads(pickle.dumps(MISSING)) is MISSING
```

Requirements:

1. `repr(MISSING)` should be `"<MISSING>"`.
2. The sentinel should remain a singleton.
3. Demonstrate that identity is preserved after pickle round-trip.

In [18]:
# Your solution here

### Solution 9

A custom `__new__` enforces one instance. A custom `__reduce__` tells pickle how to reconstruct the same singleton.

In [19]:
def get_missing():
    return MISSING

class MissingType:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __repr__(self):
        return "<MISSING>"

    def __reduce__(self):
        return (get_missing, ())


MISSING = MissingType()

round_tripped = pickle.loads(pickle.dumps(MISSING))

assert MissingType() is MISSING
assert round_tripped is MISSING
assert repr(MISSING) == "<MISSING>"

MISSING, round_tripped, round_tripped is MISSING

(<MISSING>, <MISSING>, True)

## Problem 10 — Preserve function signatures with sentinels

Write a decorator `require_at_least_one(*names)`.

It should decorate functions that use sentinel defaults. When the decorated function is called, the decorator should raise `TypeError` if all named parameters are omitted.

Example target:

```python
SKIP = object()

@require_at_least_one("name", "email")
def update_contact(*, name=SKIP, email=SKIP):
    ...
```

Calling `update_contact()` should raise `TypeError`, but calling `update_contact(name=None)` should be accepted.

Requirements:

1. Use `inspect.signature`.
2. Bind passed arguments without applying defaults.
3. Determine omission by checking whether a name is absent from the bound arguments, not by checking whether its value is `None`.
4. Preserve the wrapped function’s metadata with `functools.wraps`.

In [20]:
# Your solution here

### Solution 10

The decorator should not need to know the sentinel object. It can detect omission by checking which arguments were actually bound.

In [21]:
def require_at_least_one(*names):
    def decorate(fn):
        sig = inspect.signature(fn)

        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            bound = sig.bind_partial(*args, **kwargs)

            if not any(name in bound.arguments for name in names):
                joined = ", ".join(names)
                raise TypeError(f"At least one of {joined} must be provided")

            return fn(*args, **kwargs)

        return wrapper

    return decorate


SKIP = object()

@require_at_least_one("name", "email")
def update_contact(*, name=SKIP, email=SKIP):
    changes = {}
    if name is not SKIP:
        changes["name"] = name
    if email is not SKIP:
        changes["email"] = email
    return changes


assert update_contact(name=None) == {"name": None}
assert update_contact(email="x@example.com") == {"email": "x@example.com"}

try:
    update_contact()
except TypeError as ex:
    assert "At least one of name, email" in str(ex)
else:
    raise AssertionError("Expected TypeError")

update_contact(name=None), update_contact(email="x@example.com"), update_contact.__name__

({'name': None}, {'email': 'x@example.com'}, 'update_contact')

## Problem 11 — Best-practice comparison

For each candidate sentinel strategy below, classify it as generally safe or unsafe for production APIs, and explain why.

1. `None`
2. `-1`
3. `"__missing__"`
4. `object()` stored in a private module-level variable
5. `object()` inline in the function signature, then recovered through `fn.__defaults__`
6. private custom singleton with helpful `repr`
7. equality comparison with `==`
8. identity comparison with `is`

In [22]:
# Your notes here

### Solution 11

| Strategy | Classification | Reason |
|---|---:|---|
| `None` | Context-dependent | Safe only when `None` is not a meaningful user value. |
| `-1` | Usually unsafe | The caller can intentionally or accidentally pass `-1`. |
| `"__missing__"` | Usually unsafe | Strings are ordinary user values and can collide. |
| private module-level `object()` | Generally safe | Unique object identity is extremely unlikely to be passed accidentally. |
| inline `object()` plus `__defaults__` | Advanced / fragile | Can work, but depends on introspection and can be fragile with rebinding/decorators. |
| private custom singleton with `repr` | Strong choice | Unique identity plus better debugging and signatures. |
| `==` | Unsafe | Equality can be overloaded by user objects. |
| `is` | Correct | Sentinels are identity markers, not value markers. |

## Problem 12 — Capstone: robust configuration merge

Implement:

```python
def merge_config(base, override=..., *, retries=..., timeout=..., headers=...):
    ...
```

Rules:

1. `base` is a dictionary.
2. If `override` is omitted, start from a shallow copy of `base`.
3. If `override` is provided:
   - if it is `None`, start from an empty dictionary
   - otherwise start from `{**base, **override}`
4. Keyword-only parameters update individual fields:
   - omitted: leave as-is
   - `None`: set the field to `None`
   - any other value: set the field to that value
5. Do not mutate `base` or `override`.
6. Use one sentinel object for all omitted checks.
7. Add tests for omitted values, explicit `None`, and overrides.

In [23]:
# Your solution here

### Solution 12

This combines sentinels, update semantics, `None` as data, keyword-only parameters, and mutation avoidance.

In [24]:
CONFIG_MISSING = object()

def merge_config(base, override=CONFIG_MISSING, *, retries=CONFIG_MISSING, timeout=CONFIG_MISSING, headers=CONFIG_MISSING):
    if override is CONFIG_MISSING:
        result = base.copy()
    elif override is None:
        result = {}
    else:
        result = {**base, **override}

    if retries is not CONFIG_MISSING:
        result["retries"] = retries

    if timeout is not CONFIG_MISSING:
        result["timeout"] = timeout

    if headers is not CONFIG_MISSING:
        result["headers"] = headers

    return result


base = {"retries": 3, "timeout": 10, "headers": {"User-Agent": "demo"}}
override = {"timeout": 20, "extra": True}

assert merge_config(base) == base
assert merge_config(base) is not base

assert merge_config(base, override) == {
    "retries": 3,
    "timeout": 20,
    "headers": {"User-Agent": "demo"},
    "extra": True,
}

assert merge_config(base, None) == {}
assert merge_config(base, timeout=None)["timeout"] is None
assert merge_config(base, override, retries=0, headers=None) == {
    "retries": 0,
    "timeout": 20,
    "headers": None,
    "extra": True,
}

assert base == {"retries": 3, "timeout": 10, "headers": {"User-Agent": "demo"}}
assert override == {"timeout": 20, "extra": True}

merge_config(base), merge_config(base, override), merge_config(base, None), merge_config(base, timeout=None)

({'retries': 3, 'timeout': 10, 'headers': {'User-Agent': 'demo'}},
 {'retries': 3,
  'timeout': 20,
  'headers': {'User-Agent': 'demo'},
  'extra': True},
 {},
 {'retries': 3, 'timeout': None, 'headers': {'User-Agent': 'demo'}})

## Summary: Sentinel Best Practices

Use sentinels when you must distinguish:

1. omitted argument
2. explicit `None`
3. explicit non-`None` value

Best practices:

- prefer a private module-level sentinel for normal production code
- use `is` / `is not`, never `==`
- give custom sentinels a useful `repr`
- avoid public exposure unless intentionally designing a public API sentinel
- be careful with inline `object()` defaults and function introspection
- when decorators are involved, avoid relying on a function’s global name inside its own body
- treat sentinels as identity markers, not ordinary values